In [6]:
import sys
import importlib

sys.path.append('../utils')
import visualization
importlib.reload(visualization)
from visualization import save_to_html

import pandas as pd
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')


df = pd.read_parquet('../data/processed/telco_churn.parquet')

print(f"{df.shape[0]:,} строк и {df.shape[1]} колонок")
df.info()

7,043 строк и 27 колонок
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 27 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   customerID                    7043 non-null   string  
 1   gender                        7043 non-null   category
 2   Partner                       7043 non-null   int8    
 3   Dependents                    7043 non-null   int8    
 4   tenure                        7043 non-null   int64   
 5   PhoneService                  7043 non-null   int8    
 6   MultipleLines                 7043 non-null   category
 7   InternetService               7043 non-null   category
 8   OnlineSecurity                7043 non-null   category
 9   OnlineBackup                  7043 non-null   category
 10  DeviceProtection              7043 non-null   category
 11  TechSupport                   7043 non-null   category
 12  StreamingTV            

In [7]:
# Новые флаги
df['high_monthly_flag'] = (df['MonthlyCharges'] > 80).astype(int)
df = df.rename(columns={'SeniorCitizen': 'senior_citizen_flag'})

# Количество дополнительных услуг у клиента
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                'TechSupport', 'StreamingTV', 'StreamingMovies']

df['num_additional_services'] = df[service_cols].apply(
    lambda x: (x == 'Yes').sum(), axis=1
)

# Флаг наличия хотя бы одной дополнительной услуги
df['has_additional_services'] = (df['num_additional_services'] > 0).astype(int)

# Комбинированный рисковый признак
df['high_risk_combination'] = (
    ((df['tenure_group'] == '1-6 months') & (df['high_monthly_flag'] == 1)) | 
    ((df['tenure_group'] == '7-12 months') & (df['high_monthly_flag'] == 1)) | 
    ((df['tenure_group'] == '1-6 months') & (df['fiber_optic_flag'] == 1)) | 
    ((df['tenure_group'] == '1-6 months') & (df['senior_citizen_flag'] == 1)) | 
    ((df['tenure_group'] == '1-6 months') & (df['electronic_check_flag'] == 1)) | 
    ((df['tenure_group'] == '1-6 months') & (df['TechSupport'] == 'No')) | 
    ((df['tenure_group'] == '1-6 months') & (df['OnlineSecurity'] == 'No')) | 
    ((df['month_to_month_flag'] == 1) & (df['senior_citizen_flag'] == 1)) | 
    ((df['month_to_month_flag'] == 1) & (df['fiber_optic_flag'] == 1)) | 
    ((df['electronic_check_flag'] == 1) & (df['OnlineBackup'] == 'No')) | 
    ((df['tenure_group'] == 1) & (df['electronic_check_flag'] == 1)) | 
    ((df['electronic_check_flag'] == 1) & (df['high_monthly_flag'] == 1)) | 
    ((df['fiber_optic_flag'] == 1) & (df['high_monthly_flag'] == 1)) 
).astype(int)

In [18]:
# Проверка новых признаков

new_features = [ 
    'avg_monthly_per_tenure_month',
    'num_additional_services',
    'has_additional_services',
    'high_monthly_flag',
    'fiber_optic_flag',
    'electronic_check_flag',
    'month_to_month_flag',
    'senior_citizen_flag',
    'high_risk_combination'
]

print("Первые 5 строк новых признаков:")
display(
    df[['customerID'] + new_features].head(10)
)

features_figs = {}
features_tables = {}

for col in ['has_additional_services', 'high_risk_combination']:
    freq = df[col].value_counts()
    percent = df[col].value_counts(normalize=True) * 100
    table = pd.concat([freq, percent.round(2)], axis=1, keys=['Count', 'Percent (%)'])

    fig = px.pie(df, names=col, title=f'Распределение {col}',
             color_discrete_sequence=px.colors.qualitative.Set2,
             hole=0.4)
    fig.update_traces(textinfo='percent+label')
    fig.update_layout(height=500)

    print(f'Распределение {col}:')
    display(table)
    fig.show()

    features_figs[f'Распределение {col}'] = fig

Первые 5 строк новых признаков:


,customerID,avg_monthly_per_tenure_month,num_additional_services,has_additional_services,high_monthly_flag,fiber_optic_flag,electronic_check_flag,month_to_month_flag,senior_citizen_flag,high_risk_combination
0,7639-LIAYI,1.53,4,1,0,0,0,0,0,0
1,9803-FTJCG,0.94,3,1,0,0,0,0,0,0
2,5160-UXJED,2.62,0,0,0,0,0,0,0,0
3,6122-EFVKN,1.49,2,1,0,0,0,0,0,0
4,3583-EKAPL,55.00,1,1,0,0,1,1,0,1
5,4332-MUOEZ,4.72,4,1,1,1,0,0,1,1
6,4381-MHQDC,1.38,3,1,0,0,0,0,0,0
7,3292-PBZEJ,10.13,5,1,1,1,1,1,1,1
8,5377-NDTOU,1.28,6,1,1,0,0,0,0,0
9,8272-ONJLV,7.98,3,1,1,1,1,0,0,1


Распределение has_additional_services:


,Count,Percent (%)
has_additional_services,,
1,4824,68.49
0,2219,31.51


Распределение high_risk_combination:


,Count,Percent (%)
high_risk_combination,,
1,3946,56.03
0,3097,43.97


In [11]:
# Связь новых признаков с Churn 

for col in new_features:
    if col == 'avg_monthly_per_tenure_month':
        fig = px.histogram(df, x=col, color='Churn', 
                        nbins=50,
                        title=f'Распределение {col} по Churn',
                        color_discrete_sequence=px.colors.qualitative.Set2,
                        barmode='overlay')
        fig.update_layout(height=500)

        print(f'Описание {col} по Churn:')
        display(df.groupby('Churn')[col].describe())
        fig.show()

        features_figs[f'Распределение {col} по Churn'] = fig
        features_tables[f'Описание {col} по Churn'] = df.groupby('Churn')[col].describe().reset_index()

    else:
        churn_rate = (df.groupby(col, observed=False)['Churn']
                        .mean()
                        .mul(100)           
                        .round(2)
                        .reset_index(name='Churn_Rate'))
        
        churn_rate = churn_rate.sort_values('Churn_Rate', ascending=False)
        churn_rate_display = churn_rate[[col, 'Churn_Rate']]
        
        fig = px.bar(
            churn_rate,
            x=col,
            y='Churn_Rate',
            title=f'Процент оттока (Churn = 1) по {col}',
            color='Churn_Rate',                    
            color_continuous_scale='RdYlGn_r',       
            text='Churn_Rate',
            labels={'Churn_Rate': 'Процент ушедших (%)', col: col}        
        )
        
        fig.update_traces(texttemplate='%{y:.2f}%')
        
        print(f"\nChurn Rate по {col}:")
        display(churn_rate_display.style.hide(axis="index"))
        fig.show()

        features_figs[f'Распределение {col} по Churn'] = fig

save_to_html(
    figures_dict=features_figs, 
    tables_dict=features_tables, 
    filename="features.html", 
    title="Uni and Bivariate Analysis of New Features",
    dashboard_dir="feature_engineering"
)

Описание avg_monthly_per_tenure_month по Churn:


,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
0,5163.0,4.822287,9.857347,0.27,1.09,1.66,3.44,95.85
1,1869.0,19.112215,24.247172,0.32,2.70,7.48,24.53,102.45



Churn Rate по num_additional_services:


num_additional_services,Churn_Rate
1,45.760000
2,35.820000
3,27.370000
4,22.300000
0,21.410000
5,12.430000
6,5.280000



Churn Rate по has_additional_services:


...


Churn Rate по senior_citizen_flag:


senior_citizen_flag,Churn_Rate
1,41.680000
0,23.610000



Churn Rate по high_risk_combination:


high_risk_combination,Churn_Rate
1,41.380000
0,7.620000


'Сохранено в ../dashboards/feature_engineering/features.html'

In [10]:
df.to_csv('../data/processed/data_featured.csv', index=False)
df.to_parquet('../data/processed/data_featured.parquet', index=False)